# 5. CrewAI — Multi-Agent Collaboration

## Overview

LangChain gives you one agent with many tools. LangGraph gives you a graph you control.
**CrewAI** takes a third position: model the work the way a *team* would do it. You define
people — a researcher, an analyst, a writer — give each one a role and a goal, hand the team a
list of tasks, and let them pass work between themselves.

The team metaphor fits parts of healthcare administration unusually well, because the work is
already organised that way. A benefits question that reaches a member goes through a benefits
researcher, a coverage analyst and a communications writer in most plans. A new member-facing
tool goes past privacy, clinical governance and operations before it ships. Those are real
hand-offs between real specialists, and CrewAI lets you write them down almost literally.

In this notebook you will:

- Point CrewAI at **Amazon Bedrock**
- Write effective **roles, goals and backstories**
- Chain tasks so one agent's output becomes another's input
- Give agents **custom tools**
- Run a **hierarchical** crew where a manager agent delegates
- Capture **structured output** and inspect token usage


## Concepts

| Concept | Meaning |
|---|---|
| **Agent** | A role-playing worker: `role`, `goal`, `backstory`, optional `tools` |
| **Task** | A unit of work: `description`, `expected_output`, assigned to an agent |
| **Crew** | The team plus the task list plus a process |
| **Process** | `sequential` (tasks in order) or `hierarchical` (a manager delegates) |
| **Context** | Passing one task's output into a later task |
| **Delegation** | An agent asking a colleague for help mid-task |

> **Where CrewAI belongs in this domain — and where it does not.**
>
> Notice what this notebook does *not* do: it never puts a crew in front of a member, and it
> never lets a crew make a coverage determination. Every crew here produces **internal drafts
> and reviews that a human then approves**.
>
> That is deliberate. CrewAI's strength is readable collaboration; its weakness is that the path
> varies between runs and there is no state you can branch on or replay. For anything a member
> receives or a regulator inspects, use the LangGraph pattern from notebook 4. Use a crew where
> a human is the last step.
>
> All plan data below is invented for training.

#### 0. Setup

CrewAI reaches Bedrock through LiteLLM, which uses the standard boto3 credential chain — so on
SageMaker it picks up your **execution role** with no configuration at all.

The one thing it does need is the region under the name it expects: LiteLLM looks for
**`AWS_REGION_NAME`**, not `AWS_REGION`, so the cell below sets both.

> **Why `await ... kickoff_async()`?** Jupyter already runs an async event loop. Newer CrewAI versions refuse to run `crew.kickoff()` inside that loop and raise *"Agent execution was invoked synchronously from within a running event loop"*. Awaiting `kickoff_async()` works on both old and new CrewAI versions and returns the same result object.

In [1]:
# ---------------------------------------------------------------------------
# SageMaker Studio setup
#
# There is no .env file and no access key in this lab. boto3 automatically
# picks up short-lived credentials from the SageMaker execution role attached
# to this space, which is both simpler and safer than a long-lived key pair.
# ---------------------------------------------------------------------------
import os
import boto3

# Leave this as None to use the region this SageMaker space is running in.
# Set it ONLY if you enabled Bedrock model access in a different region.
BEDROCK_REGION = None

AWS_REGION = BEDROCK_REGION or boto3.session.Session().region_name or "us-east-1"
CHAT_MODEL_ID = os.environ.get("BEDROCK_CHAT_MODEL_ID", "amazon.nova-pro-v1:0")

session = boto3.Session(region_name=AWS_REGION)
EXECUTION_ROLE_ARN = session.client("sts").get_caller_identity()["Arn"]

# CrewAI / LiteLLM look for these exact names
os.environ["AWS_REGION_NAME"] = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION

from crewai import Agent, Task, Crew, Process, LLM

bedrock_llm = LLM(
    model=f"bedrock/{CHAT_MODEL_ID}",
    temperature=0.3,
    max_tokens=2048,
)

print("Region         :", AWS_REGION)
print("Execution role :", EXECUTION_ROLE_ARN)
print("CrewAI LLM configured for bedrock/" + CHAT_MODEL_ID)

Region         : us-east-1
Execution role : arn:aws:sts::701043845395:assumed-role/AmazonSageMaker-ExecutionRole-20261022T123794/SageMaker
CrewAI LLM configured for bedrock/amazon.nova-pro-v1:0


In [2]:
# Smoke test - confirm CrewAI can reach Bedrock before building a crew
print(bedrock_llm.call("Reply with exactly: connection ok"))

connection ok


---
## 1. A single agent and a single task

Start with the smallest possible crew: one agent, one task. Everything else in CrewAI is a
repetition of this pattern.

#### 1.1. Roles and tasks

An agent is defined by three fields, and each one does real work:

- **`role`** — the job title. Keep it specific: "Medicare Benefits Researcher", not "Helper".
- **`goal`** — what success looks like for this agent, in one sentence.
- **`backstory`** — the experience that shapes *how* it works. This is the highest-leverage
  field; it is where you encode standards, biases and method.

A task is defined by:

- **`description`** — what to do, including any `{placeholders}` filled at kickoff
- **`expected_output`** — the shape of the deliverable. Be concrete. Vague expected output is
  the single most common cause of disappointing crew results.

In [3]:
researcher = Agent(
    role="Medicare Benefits Researcher",
    goal="Find accurate, specific, plainly-stated facts about how {topic} works",
    backstory=(
        "You spent eight years in a health plan's benefits configuration team. You have "
        "watched members be harmed by confident, imprecise answers, so you never state a "
        "coverage rule without naming its conditions and exceptions. You distinguish sharply "
        "between what Original Medicare covers and what varies by plan. When you are unsure, "
        "you say so explicitly rather than filling the gap with a plausible guess. "
        "You never give clinical advice."
    ),
    llm=bedrock_llm,
    verbose=True,
    allow_delegation=False,
)

research_task = Task(
    description=(
        "Research how {topic} works under Medicare. Cover: what it is, who it applies to, "
        "the conditions and exceptions, what the member typically pays, and what varies by "
        "plan type."
    ),
    expected_output=(
        "A structured brief with exactly five sections: WHAT IT IS, WHO IT APPLIES TO, "
        "CONDITIONS AND EXCEPTIONS, MEMBER COST, WHAT VARIES BY PLAN. "
        "Each section 3-5 bullet points. No clinical advice. No marketing language."
    ),
    agent=researcher,
)

crew = Crew(agents=[researcher], tasks=[research_task], verbose=True)
print("Crew ready")

Crew ready


#### 1.2. Kick it off

`kickoff()` runs the crew. The `inputs` dictionary fills every `{placeholder}` in the agent
and task definitions.

In [4]:
result = await crew.kickoff_async(inputs={"topic": "the Part B late enrolment penalty"})

print("\n" + "=" * 70)
print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: cfa1a051-fbc9-41fc-83f4-d52bff377551                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research how the Part B late enrolment penalty works under Medicare. Cover: what it is, who it applies   │
│  to, the conditions and exceptions, what the member typically pays, and what varies by plan type.               │
│  ID: d9189914-7ddf-4ba1-aedf-066551105bb3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Medicare Benefits Researcher                                                                            │
│                                                                                                                 │
│  Task: Research how the Part B late enrolment penalty works under Medicare. Cover: what it is, who it applies   │
│  to, the conditions and exceptions, what the member typically pays, and what varies by plan type.               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Medicare Benefits Researcher                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### WHAT IT IS                                                                                                 │
│  - The Part B late enrollment penalty is a permanent increase in your monthly Medicare Part B premium.          │
│  - It is applied if you did not sign up for Medicare Part B when you were first eligible and do not qualify     │
│  for a Special Enrollment Period.                                                                               │
│  - The penalty is intended to discourage individuals from delaying enrollment in Part B to avoid paying         │
│  premiums during periods when they may not need medical services.                                               │
│                                                                                                                 │
│  ### WHO IT APPLIES TO                                                                                          │
│  - Individuals who were first eligible for Medicare Part B but did not enroll.                                  │
│  - Those who let their Part B coverage lapse and are re-enrolling after a gap.                                  │
│  - It does not apply to individuals who qualify for a Special Enrollment Period due to specific circumstances   │
│  (e.g., working past age 65 with employer coverage).                                                            │
│                                                                                                                 │
│  ### CONDITIONS AND EXCEPTIONS                                                                                  │
│  - The penalty does not apply if you qualify for a Special Enrollment Period, such as having employer coverage  │
│  through current employment.                                                                                    │
│  - If you have COBRA coverage after leaving an employer, this may also qualify you for a Special Enrollment     │
│  Period.                                                                                                        │
│  - The penalty is waived if you have End-Stage Renal Disease (ESRD) and are enrolling in Medicare within the    │
│  first 12 months of dialysis treatments.                                                                        │
│                                                                                                                 │
│  ### MEMBER COST                                                                                                │
│  - The penalty is calculated based on the number of months you were eligible for Part B but did not enroll.     │
│  - For each full 12-month period you were eligible but not enrolled, your monthly Part B premium increases by   │
│  10%.                                                                                                           │
│  - In 2023, the standard Part B premium is $164.90. If you incur a 10% penalty, your premium would increase to  │
│  approximately $181.40 per month.                                                                               │
│                                                                                                                 │
│  ### WHAT VARIES BY PLAN                                                                                        │
│  - The late enrollment penalty specifically affects the

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Research how the Part B late enrolment penalty works under Medicare. Cover: what it is, who it applies   │
│  to, the conditions and exceptions, what the member typically pays, and what varies by plan type.               │
│  Agent: Medicare Benefits Researcher                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


### WHAT IT IS
- The Part B late enrollment penalty is a permanent increase in your monthly Medicare Part B premium.
- It is applied if you did not sign up for Medicare Part B when you were first eligible and do not qualify for a Special Enrollment Period.
- The penalty is intended to discourage individuals from delaying enrollment in Part B to avoid paying premiums during periods when they may not need medical services.

### WHO IT APPLIES TO
- Individuals who were first eligible for Medicare Part B but did not enroll.
- Those who let their Part B coverage lapse and are re-enrolling after a gap.
- It does not apply to individuals who qualify for a Special Enrollment Period due to specific circumstances (e.g., working past age 65 with employer coverage).

### CONDITIONS AND EXCEPTIONS
- The penalty does not apply if you qualify for a Special Enrollment Period, such as having employer coverage through current employment.
- If you have COBRA coverage after leaving an employer, this may 

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: cfa1a051-fbc9-41fc-83f4-d52bff377551                                                                       │
│  Final Output: ### WHAT IT IS                                                                                   │
│  - The Part B late enrollment penalty is a permanent increase in your monthly Medicare Part B premium.          │
│  - It is applied if you did not sign up for Medicare Part B when you were first eligible and do not qualify     │
│  for a Special Enrollment Period.                                                                               │
│  - The penalty is intended to discourage individuals from delaying enrollment in Part B to avoid paying         │
│  premiums during periods when they may not need medical services.                                               │
│                                                                                                                 │
│  ### WHO IT APPLIES TO                                                                                          │
│  - Individuals who were first eligible for Medicare Part B but did not enroll.                                  │
│  - Those who let their Part B coverage lapse and are re-enrolling after a gap.                                  │
│  - It does not apply to individuals who qualify for a Special Enrollment Period due to specific circumstances   │
│  (e.g., working past age 65 with employer coverage).                                                            │
│                                                                                                                 │
│  ### CONDITIONS AND EXCEPTIONS                                                                                  │
│  - The penalty does not apply if you qualify for a Special Enrollment Period, such as having employer coverage  │
│  through current employment.                                                                                    │
│  - If you have COBRA coverage after leaving an employer, this may also qualify you for a Special Enrollment     │
│  Period.                                                                                                        │
│  - The penalty is waived if you have End-Stage Renal Disease (ESRD) and are enrolling in Medicare within the    │
│  first 12 months of dialysis treatments.                                                                        │
│                                                                                                                 │
│  ### MEMBER COST                                                                                                │
│  - The penalty is calculated based on the number of months you were eligible for Part B but did not enroll.     │
│  - For each full 12-month period you were eligible but not enrolled, your monthly Part B premium increases by   │
│  10%.                                                                                                           │
│  - In 2023, the standard Part B premium is $164.90. If you incur a 10% penalty, your premium would increase to  │
│  approximately $181.40 per month.                                                                               │
│                                                                                                                 │
│  ### WHAT VARIES BY PLAN                                                                                        │
│  - The late enrollment penalty specifically affects th

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

> **Read that output critically.** It is fluent, well-organised and **completely ungrounded** —
> the researcher had no tools and no source document, so every fact came from model memory.
> Compare it to the cited answers from notebook 3.
>
> This is the central trade in CrewAI: you get readable collaboration quickly, and you have to
> add grounding yourself. Section 3 attaches tools; a production version would attach a
> retrieval tool pointed at your own plan documents.

---
## 2. A multi-agent crew

Now the real pattern: three specialists, each doing the part they are good at, passing work
down the line. This mirrors how a member communication is actually produced.

```
  Researcher  ->  Coverage Analyst  ->  Communications Writer
   (finds)          (judges)                (translates)
```

The `context` parameter on a task is what makes this work — it feeds earlier task outputs into
the later task's prompt.

In [5]:
analyst = Agent(
    role="Coverage Analyst",
    goal="Turn benefits research about {topic} into a defensible position on what members should be told",
    backstory=(
        "You review benefits language for a health plan before it goes out. You have seen "
        "appeals and complaints trace back to a single over-confident sentence in a member "
        "letter. You always separate what is universally true from what depends on the "
        "member's plan, and you always name the case where the general rule does not hold."
    ),
    llm=bedrock_llm,
    verbose=True,
    allow_delegation=False,
)

writer = Agent(
    role="Member Communications Writer",
    goal="Produce a clear one-page explanation a member can read once and act on",
    backstory=(
        "You write for people who are stressed, possibly unwell, and reading on a phone. "
        "You write at a sixth-grade reading level without being condescending. You lead with "
        "what the member needs to do. You state every deadline as a date or a number of days, "
        "never as 'promptly'. You never give clinical advice, and you always tell the member "
        "how to reach a human."
    ),
    llm=bedrock_llm,
    verbose=True,
    allow_delegation=False,
)

In [6]:
task_research = Task(
    description=(
        "Research how {topic} works under Medicare. Cover the rule, its conditions, the "
        "exceptions, typical member cost, and what varies by plan type."
    ),
    expected_output="A factual brief of 250-350 words with clearly labelled sections.",
    agent=researcher,
)

task_analyse = Task(
    description=(
        "Using the research provided, decide what a member facing {topic} should actually be "
        "told. Separate what is always true from what depends on their plan. Name the "
        "situations where the general rule does not apply."
    ),
    expected_output=(
        "A position with three parts: ALWAYS TRUE, DEPENDS ON PLAN, and EXCEPTIONS TO FLAG. "
        "200 words maximum."
    ),
    agent=analyst,
    context=[task_research],       # <-- receives the researcher's output
)

task_write = Task(
    description=(
        "Turn the research and the analysis into a one-page member explanation of {topic}."
    ),
    expected_output=(
        "A markdown document with: a one-sentence summary in bold, a 'What this means for you' "
        "section, a 'What you need to do' numbered list with any deadline stated explicitly, "
        "and a closing line telling the member how to reach a person. "
        "Sixth-grade reading level. No clinical advice."
    ),
    agent=writer,
    context=[task_research, task_analyse],   # <-- receives both
)

comms_crew = Crew(
    agents=[researcher, analyst, writer],
    tasks=[task_research, task_analyse, task_write],
    process=Process.sequential,
    verbose=True,
)
print("Three-agent crew ready")

Three-agent crew ready


In [7]:
output = await comms_crew.kickoff_async(inputs={"topic": "a claim denied for missing prior authorisation"})

print("\n" + "=" * 70)
print("FINAL DELIVERABLE\n")
print(output.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: bacb6dc6-b602-41d7-af69-2ce3c53c9830                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research how a claim denied for missing prior authorisation works under Medicare. Cover the rule, its    │
│  conditions, the exceptions, typical member cost, and what varies by plan type.                                 │
│  ID: 4d84868f-f182-4f16-b558-1af55ef04957                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Medicare Benefits Researcher                                                                            │
│                                                                                                                 │
│  Task: Research how a claim denied for missing prior authorisation works under Medicare. Cover the rule, its    │
│  conditions, the exceptions, typical member cost, and what varies by plan type.                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Medicare Benefits Researcher                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Rule:**                                                                                                      │
│  Under Original Medicare (Part A and Part B), if a service requires prior authorization and it is not           │
│  obtained, the claim for that service may be denied. This means Medicare will not pay for the service, and the  │
│  member may be responsible for the full cost.                                                                   │
│                                                                                                                 │
│  **Conditions:**                                                                                                │
│  1. The service or item must be one that Medicare requires prior authorization for.                             │
│  2. The provider must have failed to obtain the necessary prior authorization before providing the service.     │
│  3. The member must have received the service from a Medicare-enrolled provider.                                │
│                                                                                                                 │
│  **Exceptions:**                                                                                                │
│  1. **Urgent or Emergency Situations:** If the service was needed urgently or in an emergency, prior            │
│  authorization may not be required.                                                                             │
│  2. **Appeals:** Members have the right to appeal a denied claim. During the appeal process, new information    │
│  or documentation may be provided to support the claim.                                                         │
│  3. **Plan-Specific Rules:** Medicare Advantage (Part C) plans may have different rules regarding prior         │
│  authorization, and members should refer to their specific plan documents.                                      │
│                                                                                                                 │
│  **Typical Member Cost:**                                                                                       │
│  If a claim is denied for missing prior authorization under Original Medicare, the member is typically          │
│  responsible for the full cost of the service. This can vary widely depending on the service provided.          │
│                                                                                                                 │
│  **What Varies by Plan Type:**                                                                                  │
│  1. **Original Medicare vs. Medicare Advantage:**                                                               │
│     - **Original Medicare:** Members are responsible for the full cost of denied services.                      │
│     - **Medicare Advantage (Part C):** These plans may have different rules for prior authorization and         │
│  cost-sharing. Members should check their plan documents for specific details.                                  │
│  2. **Medicare Part D (Prescription Drug Coverage):** Some prescription drugs may require prior authorization.  │
│  If not obtained, the claim may be denied, and the member may face higher out-of-pocket costs.                  │
│  3. **Medigap (Medicare Supplement Insurance):** These 

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Member Communications Writer                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Your Medicare Claim Was Denied for Missing Prior Authorization                                              │
│                                                                                                                 │
│  **Your Medicare claim was denied because the required prior authorization was not obtained before the service  │
│  was provided.**                                                                                                │
│                                                                                                                 │
│  ### What this means for you                                                                                    │
│                                                                                                                 │
│  If a service needs prior authorization and it is not obtained, Medicare may deny the claim. This means         │
│  Medicare will not pay for the service, and you may have to pay the full cost. This rule applies if:            │
│  1. The service requires prior authorization.                                                                   │
│  2. The provider did not get the needed prior authorization.                                                    │
│  3. You received the service from a Medicare-enrolled provider.                                                 │
│                                                                                                                 │
│  ### What you need to do                                                                                        │
│                                                                                                                 │
│  1. **Check if the situation was urgent or an emergency.** If it was, prior authorization may not have been     │
│  needed.                                                                                                        │
│  2. **Appeal the denied claim** if you believe it was wrongly denied. You can provide new information or        │
│  documentation to support your claim.                                                                           │
│     - **Deadline:** You must start the appeal process within 60 days from the date you receive the denial       │
│  notice.                                                                                                        │
│  3. **Review your Medicare plan documents** if you have a Medicare Advantage (Part C) plan or Medicare Part D   │
│  (Prescription Drug Coverage). These plans may have different rules for prior authorization.                    │
│  4. **Consider your Medigap policy** if you have one. It may help cover some costs associated with denied       │
│  claims, but coverage varies by policy.                                                                         │
│                                                                                                                 │
│  ### Talk to a person                                                                                           │
│                                                                                                                 │
│  If you have questions or need help, call Medicare at 1-800-MEDICARE (1-800-633-4227) or visit your local       │
│  Social Security office.                               

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Turn the research and the analysis into a one-page member explanation of a claim denied for missing      │
│  prior authorisation.                                                                                           │
│  Agent: Member Communications Writer                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL DELIVERABLE

## Your Medicare Claim Was Denied for Missing Prior Authorization

**Your Medicare claim was denied because the required prior authorization was not obtained before the service was provided.**

### What this means for you

If a service needs prior authorization and it is not obtained, Medicare may deny the claim. This means Medicare will not pay for the service, and you may have to pay the full cost. This rule applies if:
1. The service requires prior authorization.
2. The provider did not get the needed prior authorization.
3. You received the service from a Medicare-enrolled provider.

### What you need to do

1. **Check if the situation was urgent or an emergency.** If it was, prior authorization may not have been needed.
2. **Appeal the denied claim** if you believe it was wrongly denied. You can provide new information or documentation to support your claim. 
   - **Deadline:** You must start the appeal process within 60 days from the date you receive the denia

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: bacb6dc6-b602-41d7-af69-2ce3c53c9830                                                                       │
│  Final Output: ## Your Medicare Claim Was Denied for Missing Prior Authorization                                │
│                                                                                                                 │
│  **Your Medicare claim was denied because the required prior authorization was not obtained before the service  │
│  was provided.**                                                                                                │
│                                                                                                                 │
│  ### What this means for you                                                                                    │
│                                                                                                                 │
│  If a service needs prior authorization and it is not obtained, Medicare may deny the claim. This means         │
│  Medicare will not pay for the service, and you may have to pay the full cost. This rule applies if:            │
│  1. The service requires prior authorization.                                                                   │
│  2. The provider did not get the needed prior authorization.                                                    │
│  3. You received the service from a Medicare-enrolled provider.                                                 │
│                                                                                                                 │
│  ### What you need to do                                                                                        │
│                                                                                                                 │
│  1. **Check if the situation was urgent or an emergency.** If it was, prior authorization may not have been     │
│  needed.                                                                                                        │
│  2. **Appeal the denied claim** if you believe it was wrongly denied. You can provide new information or        │
│  documentation to support your claim.                                                                           │
│     - **Deadline:** You must start the appeal process within 60 days from the date you receive the denial       │
│  notice.                                                                                                        │
│  3. **Review your Medicare plan documents** if you have a Medicare Advantage (Part C) plan or Medicare Part D   │
│  (Prescription Drug Coverage). These plans may have different rules for prior authorization.                    │
│  4. **Consider your Medigap policy** if you have one. It may help cover some costs associated with denied       │
│  claims, but coverage varies by policy.                                                                         │
│                                                                                                                 │
│  ### Talk to a person                                                                                           │
│                                                                                                                 │
│  If you have questions or need help, call Medicare at 1-800-MEDICARE (1-800-633-4227) or visit your local       │
│  Social Security office.                              

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

#### 2.1. Inspect each agent's contribution

In [8]:
for i, task_output in enumerate(output.tasks_output, 1):
    print("=" * 70)
    print(f"TASK {i} - agent: {task_output.agent}")
    print(task_output.raw[:400].strip())
    print()

print("TOKEN USAGE:", output.token_usage)

TASK 1 - agent: Medicare Benefits Researcher
**Rule:**
Under Original Medicare (Part A and Part B), if a service requires prior authorization and it is not obtained, the claim for that service may be denied. This means Medicare will not pay for the service, and the member may be responsible for the full cost.

**Conditions:**
1. The service or item must be one that Medicare requires prior authorization for.
2. The provider must have failed t

TASK 2 - agent: Coverage Analyst
**ALWAYS TRUE:** Under Original Medicare (Part A and Part B), if a service requires prior authorization and it is not obtained, the claim for that service may be denied. This means Medicare will not pay for the service, and the member may be responsible for the full cost. The service or item must be one that Medicare requires prior authorization for, the provider must have failed to obtain the nec

TASK 3 - agent: Member Communications Writer
## Your Medicare Claim Was Denied for Missing Prior Authorization

**Your

---
## 3. Giving agents tools

Agents without tools can only reason over what is already in the prompt. Tools let them act.
CrewAI's `@tool` decorator works like LangChain's: the name and docstring tell the agent when
to use it.

This is how you stop a crew inventing dollar amounts.

In [9]:
from crewai.tools import tool


@tool("Member Cost Estimator")
def member_cost_estimator(billed_amount: float, deductible_remaining: float,
                          coinsurance_pct: float = 20.0) -> str:
    """Estimate what a member owes under a deductible-plus-coinsurance plan.

    billed_amount is the allowed amount in USD, deductible_remaining is the unmet portion
    of the annual deductible, and coinsurance_pct is the member's share after the deductible.
    Returns a sample estimate for training, not a coverage determination.
    """
    to_deductible = min(billed_amount, max(deductible_remaining, 0))
    after_deductible = billed_amount - to_deductible
    coinsurance = round(after_deductible * coinsurance_pct / 100, 2)
    total = round(to_deductible + coinsurance, 2)
    return (
        f"Allowed ${billed_amount}: ${to_deductible} to deductible, "
        f"${coinsurance} coinsurance ({coinsurance_pct}% of ${after_deductible}). "
        f"Estimated member responsibility ${total}."
    )


@tool("Prior Authorisation Checker")
def prior_auth_checker(service: str) -> str:
    """Check whether a service requires prior authorisation under the sample plan.

    Valid services: mri, ct_scan, pet_scan, inpatient_admission, outpatient_surgery,
    physical_therapy, durable_medical_equipment, primary_care_visit, lab_work.
    """
    rules = {
        "mri": "REQUIRED for outpatient MRI",
        "ct_scan": "REQUIRED for outpatient CT",
        "pet_scan": "REQUIRED",
        "inpatient_admission": "REQUIRED except emergency admissions",
        "outpatient_surgery": "REQUIRED for selected procedures",
        "physical_therapy": "Not required for the first 12 visits, then REQUIRED",
        "durable_medical_equipment": "REQUIRED for items over $500",
        "primary_care_visit": "Not required",
        "lab_work": "Not required for in-network labs",
    }
    return rules.get(service.lower().strip().replace(" ", "_"),
                     f"No rule on file for '{service}'")


print("Tools defined")

Tools defined


In [10]:
cost_agent = Agent(
    role="Member Cost Estimator",
    goal="Produce accurate cost estimates using the calculator tool, never by mental arithmetic",
    backstory=(
        "You work in a health plan's cost transparency team. You never estimate a number in "
        "your head when a tool can compute it, and you always show the inputs you used so the "
        "figure can be checked. You state clearly that an estimate is not a guarantee of "
        "payment, because you have seen members plan their finances around one."
    ),
    tools=[member_cost_estimator, prior_auth_checker],
    llm=bedrock_llm,
    verbose=True,
    allow_delegation=False,
)

cost_task = Task(
    description=(
        "A member is scheduled for an outpatient MRI with an allowed amount of $1,842. "
        "They have $400 of their annual deductible remaining and 20% coinsurance. "
        "Check whether the service needs prior authorisation, then calculate what the member "
        "should expect to pay."
    ),
    expected_output=(
        "A short itemised breakdown showing the prior authorisation requirement, the "
        "deductible portion, the coinsurance portion and the total, followed by one line "
        "stating that this is an estimate and not a guarantee of payment."
    ),
    agent=cost_agent,
)

cost_crew = Crew(agents=[cost_agent], tasks=[cost_task], verbose=True)
cost_result = await cost_crew.kickoff_async()

print("\n" + "=" * 70)
print(cost_result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 7a992f57-a813-4af8-ab06-21a5a6bc764a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: A member is scheduled for an outpatient MRI with an allowed amount of $1,842. They have $400 of their    │
│  annual deductible remaining and 20% coinsurance. Check whether the service needs prior authorisation, then     │
│  calculate what the member should expect to pay.                                                                │
│  ID: ef0cfc80-fa7b-469e-bc1f-240479c5acb3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Member Cost Estimator                                                                                   │
│                                                                                                                 │
│  Task: A member is scheduled for an outpatient MRI with an allowed amount of $1,842. They have $400 of their    │
│  annual deductible remaining and 20% coinsurance. Check whether the service needs prior authorisation, then     │
│  calculate what the member should expect to pay.                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: prior_authorisation_checker                                                                              │
│  Args: {'service': 'mri'}                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool prior_authorisation_checker executed with result: REQUIRED for outpatient MRI...
Tool member_cost_estimator executed with result: Allowed $1842.0: $400.0 to deductible, $288.4 coinsurance (20.0% of $1442.0). Estimated member responsibility $688.4....


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Member Cost Estimator                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  - Prior authorisation requirement: Required for outpatient MRI                                                 │
│  - Deductible portion: $400.0                                                                                   │
│  - Coinsurance portion: $288.4 (20.0% of $1442.0)                                                               │
│  - Total estimated member responsibility: $688.4                                                                │
│                                                                                                                 │
│  This is an estimate and not a guarantee of payment.                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: A member is scheduled for an outpatient MRI with an allowed amount of $1,842. They have $400 of their    │
│  annual deductible remaining and 20% coinsurance. Check whether the service needs prior authorisation, then     │
│  calculate what the member should expect to pay.                                                                │
│  Agent: Member Cost Estimator                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


- Prior authorisation requirement: Required for outpatient MRI
- Deductible portion: $400.0
- Coinsurance portion: $288.4 (20.0% of $1442.0)
- Total estimated member responsibility: $688.4

This is an estimate and not a guarantee of payment.


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 7a992f57-a813-4af8-ab06-21a5a6bc764a                                                                       │
│  Final Output: - Prior authorisation requirement: Required for outpatient MRI                                   │
│  - Deductible portion: $400.0                                                                                   │
│  - Coinsurance portion: $288.4 (20.0% of $1442.0)                                                               │
│  - Total estimated member responsibility: $688.4                                                                │
│                                                                                                                 │
│  This is an estimate and not a guarantee of payment.                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## 4. Orchestrating autonomous agent teams

`Process.sequential` runs tasks in the order you wrote them — you decide the workflow.
`Process.hierarchical` adds a **manager agent** that decides who does what, delegates, reviews
the results, and can send work back.

Two changes are required: set `process=Process.hierarchical`, and supply a `manager_llm`. Do
**not** assign an `agent` to the tasks — that is the manager's job.

The task below is a design review of a proposed member-facing assistant. It is the review a
real health plan would run before shipping anything like what you have built in this lab — and
the architecture described contains a deliberate set of problems for the crew to find.

In [11]:
from crewai import Agent, Task, Crew, Process

# --- Agents
security_reviewer = Agent(
    role="Cloud Security Reviewer",
    goal="Identify security and compliance gaps in AWS architectures",
    backstory=(
        "You audit AWS accounts against CIS benchmarks. You care about IAM least "
        "privilege, encryption at rest and in transit, and data residency."
    ),
    llm=bedrock_llm,
    verbose=True,
)

performance_reviewer = Agent(
    role="Performance Engineer",
    goal="Identify latency and scalability bottlenecks in AWS architectures",
    backstory=(
        "You have debugged production systems at scale. You think in terms of "
        "p99 latency, cold starts, throttling limits and regional round trips."
    ),
    llm=bedrock_llm,
    verbose=True,
)

cost_reviewer = Agent(
    role="FinOps Reviewer",
    goal="Identify cost risks and waste in AWS architectures",
    backstory=(
        "You have cut cloud bills by 40% without changing functionality. You look "
        "for idle capacity, oversized instances and per-token costs that scale badly."
    ),
    llm=bedrock_llm,
    verbose=True,
)

In [12]:
# --- Tasks ---
security_task = Task(
    description=(
        "Review the security and compliance aspects of this architecture:\n\n"
        "'A RAG chatbot for internal HR documents. Documents sit in one public-read "
        "S3 bucket. A Lambda function embeds queries with Titan and calls Claude on "
        "Bedrock in us-east-1. Employees are in India. All Lambda functions share one "
        "IAM role with AdministratorAccess. There is no caching and no request limit.'\n\n"
        "Focus only on IAM, encryption, data exposure, and compliance gaps."
    ),
    expected_output=(
        "SECURITY section only: each finding states the issue, the impact, and the fix. "
        "3-5 bullet points."
    ),
    agent=security_reviewer,
)

performance_task = Task(
    description=(
        "Review the performance and scalability aspects of the same architecture "
        "(see security review in context). Focus on latency, cold starts, throttling, "
        "and the India-to-us-east-1 round trip."
    ),
    expected_output=(
        "PERFORMANCE section only: each finding states the issue, the impact, and the fix. "
        "3-5 bullet points."
    ),
    agent=performance_reviewer,
    context=[security_task],
)

cost_task = Task(
    description=(
        "Review the cost risks of the same architecture (see prior reviews in context). "
        "Focus on token costs at scale, idle Lambda, and missing caching."
    ),
    expected_output=(
        "COST section only: each finding states the issue, the impact, and the fix. "
        "3-5 bullet points."
    ),
    agent=cost_reviewer,
    context=[security_task, performance_task],
)

consolidation_task = Task(
    description=(
        "Consolidate the three specialist reviews (security, performance, cost) from "
        "context into one unified report. Do not add new findings — only synthesise."
    ),
    expected_output=(
        "A consolidated review with three labelled sections: SECURITY, PERFORMANCE, COST. "
        "Each finding must state the issue, the impact, and the fix. "
        "End with the single highest-priority action across all three domains."
    ),
    agent=security_reviewer,
    context=[security_task, performance_task, cost_task],
)

# --- Crew ---
sequential_crew = Crew(
    agents=[security_reviewer, performance_reviewer, cost_reviewer],
    tasks=[security_task, performance_task, cost_task, consolidation_task],
    process=Process.sequential,
    verbose=True,
)

review = await sequential_crew.kickoff_async()

print("\n" + "=" * 70)
print(review.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 56fc5f40-f089-4238-a373-bbf1a0eaf201                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Review the security and compliance aspects of this architecture:                                         │
│                                                                                                                 │
│  'A RAG chatbot for internal HR documents. Documents sit in one public-read S3 bucket. A Lambda function        │
│  embeds queries with Titan and calls Claude on Bedrock in us-east-1. Employees are in India. All Lambda         │
│  functions share one IAM role with AdministratorAccess. There is no caching and no request limit.'              │
│                                                                                                                 │
│  Focus only on IAM, encryption, data exposure, and compliance gaps.                                             │
│  ID: be0cd506-9cc6-47df-b98d-9668ecc0a43d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Cloud Security Reviewer                                                                                 │
│                                                                                                                 │
│  Task: Review the security and compliance aspects of this architecture:                                         │
│                                                                                                                 │
│  'A RAG chatbot for internal HR documents. Documents sit in one public-read S3 bucket. A Lambda function        │
│  embeds queries with Titan and calls Claude on Bedrock in us-east-1. Employees are in India. All Lambda         │
│  functions share one IAM role with AdministratorAccess. There is no caching and no request limit.'              │
│                                                                                                                 │
│  Focus only on IAM, encryption, data exposure, and compliance gaps.                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Cloud Security Reviewer                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **SECURITY Section:**                                                                                          │
│                                                                                                                 │
│  - **Issue: IAM Role with AdministratorAccess**                                                                 │
│    - **Impact:** The Lambda functions are using an IAM role with `AdministratorAccess`, which grants full       │
│  access to all AWS services. This violates the principle of least privilege and increases the risk of           │
│  unauthorized actions and potential data breaches.                                                              │
│    - **Fix:** Create specific IAM roles for each Lambda function with the minimum permissions required. For     │
│  example, the Lambda function should only have permissions to read from the S3 bucket, invoke Bedrock, and      │
│  perform necessary logging.                                                                                     │
│                                                                                                                 │
│  - **Issue: Public-read S3 Bucket**                                                                             │
│    - **Impact:** The S3 bucket containing HR documents is set to public-read, which means anyone with the URL   │
│  can access the documents. This exposes sensitive HR data to unauthorized users.                                │
│    - **Fix:** Change the S3 bucket policy to restrict access to only authorized users and services. Use         │
│  pre-signed URLs or generate temporary access tokens for authorized users to access the documents.              │
│                                                                                                                 │
│  - **Issue: Lack of Encryption at Rest**                                                                        │
│    - **Impact:** The S3 bucket does not mention encryption at rest, which means the HR documents are stored     │
│  unencrypted. This exposes the data to potential unauthorized access if the bucket is compromised.              │
│    - **Fix:** Enable server-side encryption (SSE) for the S3 bucket using AWS KMS or S3-managed keys to ensure  │
│  that documents are encrypted at rest.                                                                          │
│                                                                                                                 │
│  - **Issue: Lack of Encryption in Transit**                                                                     │
│    - **Impact:** There is no mention of encryption in transit for the Lambda function queries to Bedrock or     │
│  within the architecture. This exposes the data to interception during transmission.                            │
│    - **Fix:** Ensure that all communications between the Lambda function and Bedrock, as well as within the     │
│  architecture, use HTTPS or other secure protocols to encrypt data in transit.                                  │
│                                                                                                                 │
│  - **Issue: Data Residency Concerns**                                                                           │
│    - **Impact:** Employees are located in India, but th

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Cloud Security Reviewer                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Consolidated Review                                                                                        │
│                                                                                                                 │
│  #### SECURITY                                                                                                  │
│                                                                                                                 │
│  - **Issue: IAM Role with AdministratorAccess**                                                                 │
│    - **Impact:** The Lambda functions are using an IAM role with `AdministratorAccess`, which grants full       │
│  access to all AWS services. This violates the principle of least privilege and increases the risk of           │
│  unauthorized actions and potential data breaches.                                                              │
│    - **Fix:** Create specific IAM roles for each Lambda function with the minimum permissions required. For     │
│  example, the Lambda function should only have permissions to read from the S3 bucket, invoke Bedrock, and      │
│  perform necessary logging.                                                                                     │
│                                                                                                                 │
│  - **Issue: Public-read S3 Bucket**                                                                             │
│    - **Impact:** The S3 bucket containing HR documents is set to public-read, which means anyone with the URL   │
│  can access the documents. This exposes sensitive HR data to unauthorized users.                                │
│    - **Fix:** Change the S3 bucket policy to restrict access to only authorized users and services. Use         │
│  pre-signed URLs or generate temporary access tokens for authorized users to access the documents.              │
│                                                                                                                 │
│  - **Issue: Lack of Encryption at Rest**                                                                        │
│    - **Impact:** The S3 bucket does not mention encryption at rest, which means the HR documents are stored     │
│  unencrypted. This exposes the data to potential unauthorized access if the bucket is compromised.              │
│    - **Fix:** Enable server-side encryption (SSE) for the S3 bucket using AWS KMS or S3-managed keys to ensure  │
│  that documents are encrypted at rest.                                                                          │
│                                                                                                                 │
│  - **Issue: Lack of Encryption in Transit**                                                                     │
│    - **Impact:** There is no mention of encryption in transit for the Lambda function queries to Bedrock or     │
│  within the architecture. This exposes the data to interception during transmission.                            │
│    - **Fix:** Ensure that all communications between the Lambda function and Bedrock, as well as within the     │
│  architecture, use HTTPS or other secure protocols to encrypt data in transit.                                  │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Consolidate the three specialist reviews (security, performance, cost) from context into one unified     │
│  report. Do not add new findings — only synthesise.                                                             │
│  Agent: Cloud Security Reviewer                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


### Consolidated Review

#### SECURITY

- **Issue: IAM Role with AdministratorAccess**
  - **Impact:** The Lambda functions are using an IAM role with `AdministratorAccess`, which grants full access to all AWS services. This violates the principle of least privilege and increases the risk of unauthorized actions and potential data breaches.
  - **Fix:** Create specific IAM roles for each Lambda function with the minimum permissions required. For example, the Lambda function should only have permissions to read from the S3 bucket, invoke Bedrock, and perform necessary logging.

- **Issue: Public-read S3 Bucket**
  - **Impact:** The S3 bucket containing HR documents is set to public-read, which means anyone with the URL can access the documents. This exposes sensitive HR data to unauthorized users.
  - **Fix:** Change the S3 bucket policy to restrict access to only authorized users and services. Use pre-signed URLs or generate temporary access tokens for authorized users to access the d

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 56fc5f40-f089-4238-a373-bbf1a0eaf201                                                                       │
│  Final Output: ### Consolidated Review                                                                          │
│                                                                                                                 │
│  #### SECURITY                                                                                                  │
│                                                                                                                 │
│  - **Issue: IAM Role with AdministratorAccess**                                                                 │
│    - **Impact:** The Lambda functions are using an IAM role with `AdministratorAccess`, which grants full       │
│  access to all AWS services. This violates the principle of least privilege and increases the risk of           │
│  unauthorized actions and potential data breaches.                                                              │
│    - **Fix:** Create specific IAM roles for each Lambda function with the minimum permissions required. For     │
│  example, the Lambda function should only have permissions to read from the S3 bucket, invoke Bedrock, and      │
│  perform necessary logging.                                                                                     │
│                                                                                                                 │
│  - **Issue: Public-read S3 Bucket**                                                                             │
│    - **Impact:** The S3 bucket containing HR documents is set to public-read, which means anyone with the URL   │
│  can access the documents. This exposes sensitive HR data to unauthorized users.                                │
│    - **Fix:** Change the S3 bucket policy to restrict access to only authorized users and services. Use         │
│  pre-signed URLs or generate temporary access tokens for authorized users to access the documents.              │
│                                                                                                                 │
│  - **Issue: Lack of Encryption at Rest**                                                                        │
│    - **Impact:** The S3 bucket does not mention encryption at rest, which means the HR documents are stored     │
│  unencrypted. This exposes the data to potential unauthorized access if the bucket is compromised.              │
│    - **Fix:** Enable server-side encryption (SSE) for the S3 bucket using AWS KMS or S3-managed keys to ensure  │
│  that documents are encrypted at rest.                                                                          │
│                                                                                                                 │
│  - **Issue: Lack of Encryption in Transit**                                                                     │
│    - **Impact:** There is no mention of encryption in transit for the Lambda function queries to Bedrock or     │
│  within the architecture. This exposes the data to interception during transmission.                            │
│    - **Fix:** Ensure that all communications between the Lambda function and Bedrock, as well as within the     │
│  architecture, use HTTPS or other secure protocols to encrypt data in transit.                                  │
│                                                       

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Watch the verbose log: the manager reads the task, picks a specialist, passes the
work, receives the answer, and moves on. You never wrote that routing logic.

**The trade-off:** a hierarchical crew makes many more model calls than a sequential one,
because the manager reasons before and after every delegation. Use sequential when you already
know the workflow; use hierarchical when you genuinely do not.

> **Worth doing with this output.** Check whether the crew found the problems you planted, and
> which ones it missed. A crew that produces three confident sections while missing the public
> S3 bucket is a good demonstration of why this pattern needs a human as the last step. The
> review is a first pass that makes a human reviewer faster — not a substitute for one.

---
## 5. Structured output

`output_pydantic` forces a task's result into a schema — which is what you need when the crew
feeds a downstream system rather than a human.

In [14]:
privacy_reviewer = Agent(
    role="Privacy and Compliance Reviewer",
    goal="Decide whether a member-facing health plan assistant is safe to launch",
    backstory=(
        "You review member-facing systems at a US health plan before launch. You focus on "
        "protected health information (PHI): where it is stored, who can see it, how long it "
        "is kept and whether it leaves the plan's control. You block a launch when PHI could be "
        "exposed, and you always name the single most important fix first."
    ),
    llm=bedrock_llm,
    verbose=False,
    allow_delegation=False,
)

In [15]:
from pydantic import BaseModel, Field
from typing import List


class LaunchVerdict(BaseModel):
    recommendation: str = Field(description="One of: LAUNCH, LAUNCH_WITH_CHANGES, DO_NOT_LAUNCH")
    blocking_issues: List[str] = Field(description="Between 2 and 5 issues that must be fixed first")
    phi_exposure_risk: str = Field(description="One of: none, low, moderate, high")
    first_action: str = Field(description="The single highest-priority next step")


verdict_task = Task(
    description=(
        "Give a final launch verdict on the member-facing benefits chatbot reviewed earlier. "
        "Assume 40,000 members and a launch during open enrolment."
    ),
    expected_output="A structured verdict object.",
    agent=privacy_reviewer,
    output_pydantic=LaunchVerdict,
)

verdict_crew = Crew(agents=[privacy_reviewer], tasks=[verdict_task], verbose=False)
verdict_result = await verdict_crew.kickoff_async()

parsed = verdict_result.pydantic
print("Recommendation   :", parsed.recommendation)
print("PHI exposure risk:", parsed.phi_exposure_risk)
print("First action     :", parsed.first_action)
print("Blocking issues  :")
for issue in parsed.blocking_issues:
    print("   -", issue)

Recommendation   : DO_NOT_LAUNCH
PHI exposure risk: high
First action     : Implement robust authentication and authorization controls to ensure only authorized users can access PHI.
Blocking issues  :
   - Implement robust authentication and authorization controls to ensure only authorized users can access PHI.
   - Encrypt all PHI both in transit and at rest to protect against unauthorized access.
   - Establish clear data retention and disposal policies for PHI to ensure it is not kept longer than necessary.
   - Conduct regular security audits and penetration testing to identify and address vulnerabilities in the system.


---
## 6. Writing crews that actually work

Most disappointing crew output traces back to one of these five things:

| Problem | Fix |
|---|---|
| Agents produce generic text | The `backstory` is bland. Give it opinions, standards, and a method |
| Output is the wrong shape | `expected_output` is vague. Describe the format explicitly |
| Later agents ignore earlier work | You forgot `context=[earlier_task]` |
| The crew is slow and expensive | Too many agents. Three focused agents beat seven vague ones |
| Agents talk in circles | `allow_delegation=True` everywhere. Turn it off unless delegation is the point |

A rule that holds up well: **one agent per distinct skill, not per step**. If two steps need
the same expertise, that is one agent doing two tasks.

#### 6.1. One more rule for this domain

**Agreement is not verification.** A crew of three agents that all approve a member letter has
not checked it three times — they have generated three plausible approvals from the same
underlying model. Where correctness genuinely matters, verify with something that is not a
language model: a tool that computes the number, a retrieval step that cites the source, a
human who signs it off. A crew is a drafting and review accelerator, not an assurance
mechanism.

---
## Summary

- CrewAI models work as a **team of role-playing agents**
- `role` / `goal` / `backstory` shape behaviour; `backstory` matters most
- `expected_output` controls the deliverable — be specific
- `context=[...]` passes one task's output into the next
- `Process.sequential` for known workflows, `Process.hierarchical` for delegation
- Tools let agents act; `output_pydantic` makes results machine-readable
- Crews are ungrounded by default — attach tools and retrieval, or you get fluent guesses
- In this domain, put a human at the end of every crew

**Next:** notebook `6-Framework-Comparison.ipynb`, where we build the same application in all
three frameworks and compare them directly.